<a href="https://colab.research.google.com/github/Rishikesh26022006/Demand-Forecasting-Model/blob/main/Demand_Forecasting_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Name : Demand Forecasting System


#1: Install Necessary Libraries

In [ ]:
!pip install prophet streamlit plotly seaborn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

#2: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error
import math

# Set plot style
plt.style.use('ggplot')
pd.set_option('display.max_columns', None)

#3: Dataset Loading

In [ ]:
#import Dataset
from google.colab import files
uploaded=files.upload()

# Load the dataset
df = pd.read_csv('Demand_Forecastong_data.csv')

# Convert date to datetime immediately
df['date'] = pd.to_datetime(df['date'])

print("--- DATA HEAD (First 5 rows) ---")
print(df.head())

print("\n--- DATA TAIL (Last 5 rows) ---")
print(df.tail())

print("\n--- DATA INFO ---")
print(df.info())

#4: Data Preprocessing

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

print("--- 1. MISSING VALUES ---")
missing_values = df.isnull().sum()
print(missing_values)

print("\n--- 2. DUPLICATES ---")
duplicates = df.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print("Duplicates dropped.")

print("\n--- 3. UNIQUE VALUES ---")
print(f"Unique Stores: {df['store'].unique()}")
print(f"Unique Items: {df['item'].nunique()}")

print("\n--- 4. OUTLIER DETECTION & HANDLING ---")

# A. Calculate Z-Scores
mean_sales = df['sales'].mean()
std_sales = df['sales'].std()
threshold = 3
outliers = df[(df['sales'] - mean_sales) / std_sales > threshold]
print(f"Found {len(outliers)} outlier records (extreme sales spikes).")

# B. Visualization BEFORE Handling
plt.figure(figsize=(15, 5))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['sales'])
plt.title('BEFORE: Sales Distribution (With Outliers)')
plt.xlabel('Sales Quantity')

# C. Handle Outliers (Method: Capping)

cap_value = mean_sales + (threshold * std_sales)
print(f"Capping all sales values greater than {cap_value:.2f}")

# Apply Capping
df['sales_clean'] = np.where(df['sales'] > cap_value, cap_value, df['sales'])

# D. Visualization AFTER Handling
plt.subplot(1, 2, 2)
sns.boxplot(x=df['sales_clean'])
plt.title('AFTER: Sales Distribution (Outliers Capped)')
plt.xlabel('Sales Quantity')

plt.tight_layout()
plt.show()

# Update the main 'sales' column to use the clean data
df['sales'] = df['sales_clean']
df.drop(columns=['sales_clean'], inplace=True)
print("Outliers handled. 'sales' column updated.")

#5: Exploratory Data Analysis (EDA)

In [ ]:
# Set figure size
plt.figure(figsize=(15, 10))

# Plot 1: Overall Sales Distribution
plt.subplot(2, 2, 1)
sns.histplot(df['sales'], bins=50, kde=True)
plt.title('Distribution of Sales Quantity')

# Plot 2: Sales Over Time (Aggregated)
daily_sales = df.groupby('date')['sales'].sum().reset_index()
plt.subplot(2, 2, 2)
plt.plot(daily_sales['date'], daily_sales['sales'], color='blue', alpha=0.6)
plt.title('Total Daily Sales Over Time')

# Plot 3: Sales by Month (Seasonality Check)
df['month'] = df['date'].dt.month
plt.subplot(2, 2, 3)
sns.boxplot(x='month', y='sales', data=df)
plt.title('Sales Seasonality by Month')

# Plot 4: Sales by Store
plt.subplot(2, 2, 4)
sns.barplot(x='store', y='sales', data=df)
plt.title('Average Sales by Store')

plt.tight_layout()
plt.show()

#6: Model Training & Evaluation

In [ ]:
# Filter for just one item to demonstrate evaluation
item_df = df[(df['store']==1) & (df['item']==1)].copy()
prophet_df = item_df[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})

# Split Train/Test
train = prophet_df[prophet_df['ds'] < '2017-01-01']
test = prophet_df[prophet_df['ds'] >= '2017-01-01']

# Train Model
m = Prophet(yearly_seasonality=True, weekly_seasonality=True)
m.fit(train)

# Predict
forecast = m.predict(test[['ds']])

# Evaluate
mae = mean_absolute_error(test['y'], forecast['yhat'])
print(f"--- MODEL EVALUATION ---")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print("This means our prediction is usually off by about this many units.")

# Plot Evaluation
plt.figure(figsize=(12, 6))
plt.plot(train['ds'], train['y'], label='Training Data')
plt.plot(test['ds'], test['y'], label='Actual Sales (Test)')
plt.plot(test['ds'], forecast['yhat'], label='Predicted Sales', alpha=0.7)
plt.legend()
plt.title('Forecast vs Actual Analysis (Model Validation)')
plt.show()

#7:The Streamlit Dashboard Product

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from prophet import Prophet
from prophet.plot import plot_plotly
from plotly import graph_objs as go
import numpy as np

# --- PAGE CONFIGURATION ---
st.set_page_config(page_title="Demand Intelligence System", layout="wide")

st.title("📊 End-to-End Demand Forecasting System")
st.markdown("### AI-Driven Inventory & Revenue Planning")

# --- 1. DATA LOADING & PROCESSING ---
@st.cache_data
def load_data():
    df = pd.read_csv('Demand_Forecastong_data.csv')
    df['date'] = pd.to_datetime(df['date'])
    return df

try:
    data = load_data()
except Exception as e:
    st.error(f"Error loading data: {e}")
    st.stop()

# --- 2. SIDEBAR CONFIGURATION ---
with st.sidebar:
    st.header("🕹️ Forecast Settings")
    stores = sorted(data['store'].unique())
    items = sorted(data['item'].unique())

    selected_store = st.selectbox("Select Store", stores)
    selected_item = st.selectbox("Select Item", items)
    days_to_predict = st.slider("Days to Forecast", 7, 365, 90)

# Filter Data
df_filtered = data[(data['store'] == selected_store) & (data['item'] == selected_item)]
df_prophet = df_filtered[['date', 'sales']].rename(columns={'date': 'ds', 'sales': 'y'})

# --- 3. DASHBOARD METRICS ---
total_sales = df_filtered['sales'].sum()
avg_sales = df_filtered['sales'].mean()

c1, c2, c3 = st.columns(3)
c1.metric("💰 Total Historical Sales", f"{total_sales:,.0f}")
c2.metric("📊 Daily Avg Sales", f"{avg_sales:.1f}")
c3.metric("📅 Record Count", f"{len(df_filtered)}")

st.markdown("---")

# --- 4. VISUALIZATION & MODELING ---
tab1, tab2, tab3 = st.tabs(["📈 Historical Analysis", "🤖 Generate Forecast", "📋 Action Plan"])

with tab1:
    st.subheader(f"Historical Performance: Store {selected_store}, Item {selected_item}")
    fig_raw = go.Figure()
    fig_raw.add_trace(go.Scatter(x=df_prophet['ds'], y=df_prophet['y'], name='Actual Sales', line=dict(color='#007BFF')))
    fig_raw.layout.update(title_text='Sales History', xaxis_rangeslider_visible=True, template='plotly_white')
    st.plotly_chart(fig_raw, use_container_width=True)

with tab2:
    if st.button("Run AI Forecast"):
        with st.spinner("Training Prophet Model on Live Data..."):
            m = Prophet(daily_seasonality=False, yearly_seasonality=True, weekly_seasonality=True)
            m.fit(df_prophet)
            future = m.make_future_dataframe(periods=days_to_predict)
            forecast = m.predict(future)

            st.success("Analysis Complete!")

            st.markdown("### 1. Future Demand Prediction")
            fig_cast = plot_plotly(m, forecast)
            st.plotly_chart(fig_cast, use_container_width=True)

            st.markdown("### 2. Decomposition (Trend & Seasonality)")
            fig_comp = m.plot_components(forecast)
            st.write(fig_comp)

            st.session_state['forecast'] = forecast
            st.session_state['has_run'] = True

with tab3:
    if 'has_run' in st.session_state:
        forecast = st.session_state['forecast']
        next_week = forecast.tail(days_to_predict).head(7)
        total_predicted = int(next_week['yhat'].sum())
        max_required = int(next_week['yhat_upper'].sum())

        st.subheader("📢 Decision Support System")
        col1, col2, col3 = st.columns(3)
        col1.metric("Predicted Sales (Next 7 Days)", f"{total_predicted} units")
        col2.metric("Recommended Inventory", f"{max_required} units")
        col3.metric("Growth Trend", "Stable/Up" if forecast['trend'].iloc[-1] > forecast['trend'].iloc[-2] else "Declining")

        st.markdown(f"""
        ### 📝 Action Plan:
        1.  **Inventory:** Order **{max_required} units** to cover next week's demand.
        2.  **Staffing:** Prepare for a total volume of {total_predicted} sales.
        """)

        csv = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(days_to_predict).to_csv(index=False).encode('utf-8')
        st.download_button("Download Forecast CSV", csv, "forecast.csv", "text/csv")
    else:
        st.info("Please go to the 'Generate Forecast' tab and run the model first.")

In [ ]:
import os
import time

# 1. Kill all old processes
print("Stopping old processes...")
os.system("pkill streamlit")
os.system("pkill cloudflared")
time.sleep(2)

# 2. Run Streamlit with SPECIAL FLAGS to fix the "Module" error
# We disable CORS and XSRF protection so the files load correctly through the tunnel
print("Starting Streamlit with fix...")
!streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false &>/dev/null&

# 3. Wait for it to boot
time.sleep(5)

# 4. Start Tunnel
print("Starting Cloudflare Tunnel...")
print("=========================================================")
print("👇 RIGHT CLICK THIS LINK -> OPEN IN INCOGNITO WINDOW 👇")
print("=========================================================")
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501